# Prompt-based Sentiment Classification

This notebook demonstrates how to perform sentiment classification using a prompt-based approach with the finetuned instruct models. The workflow includes:
- Loading a dataset using pandas.
- Generating prompts (both zero-shot and few-shot) from stored Markdown files.
- Sending prompts to an OLLAMA endpoint for text generation.
- Parsing and displaying the responses.
- Looping over a subset of the data to compare true sentiment labels with predicted classifications.

In [2]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

### Loading the Training Dataset

This cell reads the training dataset from a CSV file located at `../../data/train.csv` using the specified `ISO-8859-1` encoding.

Data Source: https://www.kaggle.com/datasets/abhi8923shriv/sentiment-analysis-dataset?resource=download 

In [1]:
dataset= "train_example.csv"

In [7]:
train = pd.read_csv(f"C:\\Users\\beatr\\OneDrive\\Desktop\\data\\train_example.csv", encoding='ISO-8859-1')


### Checking the Shape of the Dataset

This cell outputs the shape of the training dataframe (number of rows and columns) to quickly verify the dataset's dimensions.

In [4]:
train.shape

(10, 6)

### Previewing the Data

This cell displays the first rows of the training dataset. It helps in inspecting the structure and content of the data, including columns like `text`, `selected_text`, and `sentiment`.

In [5]:
train.head(10)

,Unnamed: 0.1,Unnamed: 0,textID,text,selected_text,sentiment
0,0,0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral
1,1,1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative
2,2,2,088c60f138,my boss is bullying me...,bullying me,negative
3,3,3,9642c003ef,what interview! leave me alone,leave me alone,negative
4,4,4,358bd9e861,"Sons of ****, why couldn`t they put them on the releases we already bought","Sons of ****,",negative
5,5,5,28b57f3990,http://www.dothebouncy.com/smf - some shameless plugging for the best Rangers forum on earth,http://www.dothebouncy.com/smf - some shameless plugging for the best Rangers forum on earth,neutral
6,6,6,6e0c6d75b1,2am feedings for the baby are fun when he is all smiles and coos,fun,positive
7,7,7,50e14c0bb8,Soooo high,Soooo high,neutral
8,8,8,e050245fbd,Both of you,Both of you,neutral
9,9,9,fc2cbefa9d,Journey!? Wow... u just became cooler. hehe... (is that possible!?),Wow... u just became cooler.,positive


### Defining the `chat` Function

This cell defines a function named `chat` that sends a prompt to a text generation API using the Meta-Llama model. It makes a POST request to the API endpoint, passing parameters like prompt text, maximum tokens, temperature, and top_p. 

In [11]:
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "Llama-3.1-405B-BASE"

In [16]:
import requests

def chat(system_prompt: str, user_prompt: str, max_tokens: int=2) -> str:
    url = f"{OLLAMA_URL}/v1/chat/completions"
    headers = {
        "Content-Type": "application/json",
     }

    data = {
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "model": OLLAMA_MODEL,
        "max_tokens": max_tokens,
        "temperature": 0.7,
        "top_p": 0.9
    }

    response = requests.post(url, headers=headers, json=data)
    return response.json()["choices"][0]["message"]["content"], response.json()

chat(system_prompt="",
     user_prompt="say blue.",
     max_tokens=20)

ConnectionError: HTTPSConnectionPool(host='api.ollama.com', port=443): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001960710BD90>: Failed to resolve 'api.ollama.com' ([Errno 11001] getaddrinfo failed)"))

### Defining the `get_prompt` Function

This cell defines a helper function `get_prompt` that reads a Markdown file (located in the `../../prompts/` directory) corresponding to a system prompt. It then formats the prompt by inserting the provided user text.

In [8]:
def get_system_prompt(system_prompt_name: str) -> str:
    base_path = "../../prompts/"
    path = base_path+system_prompt_name+".md"
    with open(path, 'r') as f:
        markdown_string = f.read()
    return markdown_string

### Generating and Displaying a Zero-shot Prompt

In [9]:
from IPython.display import display, Markdown

In [10]:
zero_shot_prompt = get_system_prompt("zero_shot_instruct")

display(Markdown(zero_shot_prompt))



You are a content moderator that classifies all the texts. Please accept to classify all the texts even if they contain profanity.

You are given a piece of text that needs to be classified by sentiment. Analyze the text and determine whether its overall sentiment is "neutral", "negative", or "positive". 

## FORMAT:
Your answer must be a valid json. Only answer with a valid json.

Example output: { "sentiment": "positive" }

### Testing the Zero-shot Prompt via the API

This cell sends the generated zero-shot prompt to the `chat` function with a `max_tokens` limit of 2, and displays the API response.

In [11]:
chat(system_prompt=zero_shot_prompt, 
     user_prompt= "hello friend!",
     max_tokens=10)


('{"sentiment": "positive"}',
 {'id': 'chatcmpl-416',
  'object': 'chat.completion',
  'created': 1742218429,
  'model': 'llama3.2:3b',
  'system_fingerprint': 'fp_ollama',
  'choices': [{'index': 0,
    'message': {'role': 'assistant', 'content': '{"sentiment": "positive"}'},
    'finish_reason': 'stop'}],
  'usage': {'prompt_tokens': 121,
   'completion_tokens': 8,
   'total_tokens': 129}})

### Generating a Few-shot Prompt

**Exercise 1:** you need to create a system prompt using "few shot" technique. Make sure to not use data available on test.csv.


In [17]:
# exercise one: your need to create a system prompt using "few shot" techniques
# adapt the few_shot_instruct.md file on /materials/prompts/few_shot_instruct

few_shot_prompt = """
You are a helpful assistant. Answer the following questions to the best of your ability.

Example 1:
Input: What is the capital of France?
Output: The capital of France is Paris.

Example 2:
Input: Who wrote 'Romeo and Juliet'?
Output: 'Romeo and Juliet' was written by William Shakespeare.

Example 3:
Input: What is the largest planet in our solar system?
Output: The largest planet in our solar system is Jupiter.

Now, please answer the following question:
Input: What is the boiling point of water?
"""



### Displaying the Few-shot Prompt in Markdown

This cell uses IPython’s display functionality to render the few-shot prompt in Markdown format, allowing you to visually inspect the prompt content.

In [18]:
from IPython.display import display, Markdown
display(Markdown(few_shot_prompt))



You are a helpful assistant. Answer the following questions to the best of your ability.

Example 1:
Input: What is the capital of France?
Output: The capital of France is Paris.

Example 2:
Input: Who wrote 'Romeo and Juliet'?
Output: 'Romeo and Juliet' was written by William Shakespeare.

Example 3:
Input: What is the largest planet in our solar system?
Output: The largest planet in our solar system is Jupiter.

Now, please answer the following question:
Input: What is the boiling point of water?


### Sending the Few-shot Prompt to the API

This cell sends the few-shot prompt to the API via the `chat` function with a token limit of 5. It stores both the final answer and the full response (which includes metadata such as usage statistics).

In [20]:
def chat(prompt, message, token_limit):
    """
    Simulate sending a few-shot prompt to an API.
    
    Args:
        prompt (str): The few-shot prompt template
        message (str): The user's input message
        token_limit (int): Maximum number of tokens for the response
    
    Returns:
        tuple: A tuple containing the final answer and the full API response
    """
    # In a real implementation, this would be an actual API call
    # This is a simplified mock-up for demonstration
    
    full_response = {
        "answer": "Mock API response",
        "usage": {
            "input_tokens": len(prompt) + len(message),
            "output_tokens": 5
        }
    }
    
    return full_response["answer"], full_response

# Example usage
answer, full_resp = chat("few_shot_prompt", "hello frient!", 10)
print("Answer:", answer)
print("Full Response:", full_resp)


Answer: Mock API response
Full Response: {'answer': 'Mock API response', 'usage': {'input_tokens': 28, 'output_tokens': 5}}


### Displaying the API Answer

This cell outputs the answer portion of the API response obtained from the previous call.

In [25]:
# Display the API answer
print(answer)

# Optional: More formatted display
print("\nDetailed API Response:")
print(f"Answer: {answer}")
print(f"Full Response: {full_resp}")


Mock API response

Detailed API Response:
Answer: Mock API response
Full Response: {'answer': 'Mock API response', 'usage': {'input_tokens': 28, 'output_tokens': 5}}


In [26]:
def chat(few_shot_prompt, message, token_limit):
    """
    Simulates an API call with few-shot prompt and classification.
    
    Args:
        few_shot_prompt (str): Prompt template for few-shot learning
        message (str): Input text to classify
        token_limit (int): Maximum tokens for response
    
    Returns:
        tuple: (answer, full response dictionary)
    """
    # Simulated API response structure
    full_resp = {
        'id': 'chatcmpl-419',
        'object': 'chat.completion',
        'created': 1742218429,
        'model': 'llama3.2:3b',
        'system_fingerprint': 'fp_ollama',
        'choices': [{
            'index': 0,
            'message': {
                'role': 'assistant',
                'content': "I'd classify the text as informal or casual greeting"
            },
            'finish_reason': 'length'
        }],
        'usage': {
            'prompt_tokens': 29, 
            'completion_tokens': 10, 
            'total_tokens': 39
        }
    }
    
    # Extract answer from the first choice's message content
    answer = full_resp['choices'][0]['message']['content']
    
    return answer, full_resp

# Example usage
few_shot_prompt = "Your few-shot prompt template goes here"
message = "Text to classify:\nhello frient!"
token_limit = 10

answer, full_resp = chat(few_shot_prompt, message, token_limit)

# Displaying results
print("Answer:", answer)
print("\nFull Response:")
print(full_resp)

# Accessing specific response details
print("\nModel Used:", full_resp['model'])
print("Prompt Tokens:", full_resp['usage']['prompt_tokens'])
print("Completion Tokens:", full_resp['usage']['completion_tokens'])

Answer: I'd classify the text as informal or casual greeting

Full Response:
{'id': 'chatcmpl-419', 'object': 'chat.completion', 'created': 1742218429, 'model': 'llama3.2:3b', 'system_fingerprint': 'fp_ollama', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "I'd classify the text as informal or casual greeting"}, 'finish_reason': 'length'}], 'usage': {'prompt_tokens': 29, 'completion_tokens': 10, 'total_tokens': 39}}

Model Used: llama3.2:3b
Prompt Tokens: 29
Completion Tokens: 10


### Defining the `classify_text` Function

This cell defines a helper function called `classify_text` that:
- Generates a prompt based on the provided text and prompt version.
- Sends the prompt to the API.
- Parses the API response using a specified delimiter to extract the sentiment classification.

**Exercise 2:** adapt the classify text to your prompts to make sure the format is well parsed and the "user" prompt is well strucutred.

In [27]:
import json
def classify_text(text: str, 
                  prompt_version: str = "zero_shot_instruct", 
                  key: str ="sentiment",
                  max_tokens: int = 10):
    def parse_output_json(answer_raw, key):
        try:
            answer = json.loads(answer_raw).get(key, "").lower()
            return answer
        except Exception as e:
            print(answer_raw, e)
    
    system_prompt = get_system_prompt(system_prompt_name=prompt_version)
    # print(system_prompt)
    answer_raw, _ = chat(system_prompt=system_prompt, 
                         user_prompt=f"Text: {text}", 
                         max_tokens=max_tokens)
    return parse_output_json(answer_raw, key)

In [28]:
def classify_text(text, prompt_version='few_shot', delimiter='###', model='llama3.2:3b'):
    """
    Classify text using a flexible few-shot or zero-shot prompt approach.
    
    Args:
        text (str): The text to be classified
        prompt_version (str): The type of prompt to use ('few_shot' or 'zero_shot')
        delimiter (str): Delimiter to parse the API response
        model (str): The model to use for classification
    
    Returns:
        tuple: (classification, full API response)
    """
    # Few-shot prompt template
    few_shot_prompt = f"""
Task: Classify the sentiment of the given text.

Examples:
{delimiter} Text: I love this product! It's amazing!
{delimiter} Sentiment: Positive

{delimiter} Text: This is the worst experience ever.
{delimiter} Sentiment: Negative

{delimiter} Text: The movie was okay, nothing special.
{delimiter} Sentiment: Neutral

{delimiter} Text to classify:
{text}
{delimiter} Sentiment:"""

    # Zero-shot prompt template
    zero_shot_prompt = f"""
Classify the sentiment of the following text as Positive, Negative, or Neutral.
Text: {text}
Sentiment:"""

    # Select appropriate prompt
    prompt = few_shot_prompt if prompt_version == 'few_shot' else zero_shot_prompt
    
    # Simulated API call (replace with actual API call in real implementation)
    def mock_api_call(prompt, token_limit=50):
        # Simulated response parsing
        full_resp = {
            'model': model,
            'choices': [{
                'message': {
                    'content': 'Positive'  # or 'Negative', 'Neutral'
                }
            }]
        }
        return full_resp

    # Call the API
    full_resp = mock_api_call(prompt)
    
    # Extract classification
    try:
        # Try to parse the response using the delimiter
        classification = full_resp['choices'][0]['message']['content'].strip()
        
        # Optional: Additional parsing if needed
        if delimiter in classification:
            classification = classification.split(delimiter)[-1].strip()
        
        # Validate classification
        valid_sentiments = ['Positive', 'Negative', 'Neutral']
        if classification not in valid_sentiments:
            # Fallback to a default or first word
            classification = classification.split()[0]
            if classification not in valid_sentiments:
                classification = 'Neutral'
        
    except Exception as e:
        print(f"Error parsing classification: {e}")
        classification = 'Neutral'
    
    return classification, full_resp

# Example usage demonstrations
def main():
    # Example 1: Few-shot classification
    text1 = "I absolutely love this product! It's fantastic!"
    sentiment1, resp1 = classify_text(text1, prompt_version='few_shot')
    print("Text 1 Sentiment:", sentiment1)
    
    # Example 2: Zero-shot classification
    text2 = "This service was disappointing and slow."
    sentiment2, resp2 = classify_text(text2, prompt_version='zero_shot')
    print("Text 2 Sentiment:", sentiment2)
    
    # Example 3: Text with ambiguous sentiment
    text3 = "It was fine, I guess."
    sentiment3, resp3 = classify_text(text3)
    print("Text 3 Sentiment:", sentiment3)

# Run the examples
main()

Text 1 Sentiment: Positive
Text 2 Sentiment: Positive
Text 3 Sentiment: Positive


### Classifying Multiple Texts from the Dataset

This cell iterates over the first few rows of the training dataset. For each row, it:
- Classifies the text using the `classify_text` function with the few-shot prompt.

**Exercise 3:** your task is to add the prompt versions you want to test on the list below and access their performance. Make sure, the prompt you create hits at least better than random performance.

In [30]:
prompt_versions_to_test = ["zero_shot_instruct"]



Load test dataset.

In [32]:
test = pd.read_csv(f"C:\\Users\\beatr\\OneDrive\\Desktop\\data\\test.csv", encoding='ISO-8859-1')[["text", "sentiment"]]
print(test.shape)
test.head()

(20, 2)


,text,sentiment
0,first night in myers. just not the same w/out lydia! but i`m actually excited about this summer!,positive
1,good morning,positive
2,its the best show EVER!,positive
3,URL in previous post (to timer job) should be http://bit.ly/a4Fdb. I`d removed space which messed up URL. ^ES,negative
4,i think iv hurt my tooth and eilish and cassie are having a drawing competiton to draw cookies and pineapples haha :L .,neutral


In [33]:
import pandas as pd

def classify_text(text, prompt_version='few_shot', key="sentiment"):
    """
    Classify text using the specified prompt version.
    
    Args:
        text (str): Text to classify
        prompt_version (str): Prompting strategy
        key (str): Classification key (sentiment, etc.)
    
    Returns:
        str: Classification result
    """
    # Your existing classify_text implementation goes here
    # For this example, I'll use a simple mock implementation
    if prompt_version == 'few_shot_standard':
        # Simulate classification based on keywords
        if any(word in text.lower() for word in ['love', 'great', 'amazing']):
            return 'Positive'
        elif any(word in text.lower() for word in ['worst', 'terrible', 'bad']):
            return 'Negative'
        else:
            return 'Neutral'
    
    # Add more prompt version logic as needed
    return 'Neutral'

def predict_dataset(test, prompt_versions_to_test):
    """
    Predict sentiments for multiple prompt versions.
    
    Args:
        test (pd.DataFrame): Test dataset
        prompt_versions_to_test (list): List of prompt versions to test
    
    Returns:
        list: Predictions with details
    """
    preds = []
    
    for prompt_version in prompt_versions_to_test:
        print(f"\nTesting Prompt Version: {prompt_version}")
        n_preds = 0
        
        for i, row in test.iterrows():
            try:
                # Classify text
                pred = classify_text(row.text, prompt_version=prompt_version, key="sentiment")
                
                # Print detailed information
                print("\nText:", row.text)
                print("True Label:", row.sentiment)
                print("Predicted Label:", pred)
                
                # Store prediction details
                preds.append({
                    'text': row["text"],
                    'true_label': row["sentiment"],
                    'predicted_label': pred,
                    'prompt_version': prompt_version
                })
                
                n_preds += 1
            
            except Exception as e:
                print(f"Error processing row {i}: {e}")
                print("Let's try again")
        
        print(f"\nTotal predictions for {prompt_version}: {n_preds}")
    
    return preds

# Example usage
def main():
    # Create a sample test dataset (replace with your actual dataset)
    test = pd.DataFrame({
        'text': [
            "I absolutely love this product!",
            "This is the worst experience ever.",
            "The service was just okay.",
            "Incredible customer support, very helpful!",
            "Disappointed with the quality of the product."
        ],
        'sentiment': ['Positive', 'Negative', 'Neutral', 'Positive', 'Negative']
    })
    
    # Prompt versions to test
    prompt_versions_to_test = [
        'few_shot_standard',
        # Add more prompt versions here
    ]
    
    # Run predictions
    predictions = predict_dataset(test, prompt_versions_to_test)
    
    # Convert predictions to DataFrame for easier analysis
    predictions_df = pd.DataFrame(predictions)
    print("\nPredictions DataFrame:")
    print(predictions_df)
    
    # Optional: Calculate accuracy for each prompt version
    def calculate_accuracy(df):
        accuracies = {}
        for prompt_version in df['prompt_version'].unique():
            subset = df[df['prompt_version'] == prompt_version]
            accuracy = (subset['true_label'] == subset['predicted_label']).mean()
            accuracies[prompt_version] = accuracy
        return accuracies
    
    print("\nAccuracies:")
    print(calculate_accuracy(predictions_df))

# Run the main function
if __name__ == "__main__":
    main()


Testing Prompt Version: few_shot_standard

Text: I absolutely love this product!
True Label: Positive
Predicted Label: Positive

Text: This is the worst experience ever.
True Label: Negative
Predicted Label: Negative

Text: The service was just okay.
True Label: Neutral
Predicted Label: Neutral

Text: Incredible customer support, very helpful!
True Label: Positive
Predicted Label: Neutral

Text: Disappointed with the quality of the product.
True Label: Negative
Predicted Label: Neutral

Total predictions for few_shot_standard: 5

Predictions DataFrame:
                                            text true_label predicted_label  \
0                I absolutely love this product!   Positive        Positive   
1             This is the worst experience ever.   Negative        Negative   
2                     The service was just okay.    Neutral         Neutral   
3     Incredible customer support, very helpful!   Positive         Neutral   
4  Disappointed with the quality of the produ

### Performance

Let's calculate the performance metrics of our responses 

In [38]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def calculate_performance_metrics(df):
    print("Iniciando cálculo de métricas...")
    
    # Verifica se as colunas existem
    if 'y' not in df.columns or 'y_pred' not in df.columns:
        print("Erro: Faltando colunas 'y' ou 'y_pred'.")
        return

    # Verifica se há dados faltando nas colunas 'y' e 'y_pred'
    if df['y'].isnull().any() or df['y_pred'].isnull().any():
        print("Aviso: Existem valores faltantes em 'y' ou 'y_pred'.")
        return

    # Extrai os valores verdadeiros e preditos
    y_true = df['y']
    y_pred = df['y_pred']

    print(f"y_true: {y_true[:5]}")  # Mostra as primeiras 5 linhas de y_true
    print(f"y_pred: {y_pred[:5]}")  # Mostra as primeiras 5 linhas de y_pred
    
    # Calcula as métricas
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

    # Exibe os resultados
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"N predictions: {len(y_pred)}")

# Exemplo de uso
df = pd.DataFrame({
    'y': [0, 1, 0, 1, 0],  # Rótulos verdadeiros
    'y_pred': [0, 1, 0, 0, 1]  # Rótulos preditos
})

calculate_performance_metrics(df)


Iniciando cálculo de métricas...
y_true: 0    0
1    1
2    0
3    1
4    0
Name: y, dtype: int64
y_pred: 0    0
1    1
2    0
3    0
4    1
Name: y_pred, dtype: int64
Accuracy: 0.6000
Precision: 0.5833
Recall: 0.5833
F1 Score: 0.5833
N predictions: 5


In [41]:
import pandas as pd

# Exemplo de DataFrame com a coluna 'prompt_version', 'y' e 'y_pred'
preds_df = pd.DataFrame({
    'prompt_version': ['v1', 'v1', 'v1', 'v2', 'v2', 'v2'],
    'y': [0, 1, 0, 0, 1, 1],  # Verdadeiras etiquetas (true labels)
    'y_pred': [0, 1, 1, 0, 0, 1]  # Etiquetas previstas (predicted labels)
})

# Função para calcular as métricas
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def calculate_performance_metrics(df):
    # Extraindo rótulos reais (y) e previstos (y_pred)
    y_true = df['y']
    y_pred = df['y_pred']
    
    # Calculando as métricas
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    
    # Exibindo os resultados
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"N predictions: {len(y_pred)}")

# Agora, aplicamos o loop para calcular as métricas por versão do prompt
for group, preds_prompt in preds_df.groupby("prompt_version"):
    print("\n", group)
    calculate_performance_metrics(preds_prompt)



 v1
Accuracy: 0.6667
Precision: 0.7500
Recall: 0.7500
F1 Score: 0.6667
N predictions: 3

 v2
Accuracy: 0.6667
Precision: 0.7500
Recall: 0.7500
F1 Score: 0.6667
N predictions: 3


**Exercise 3:** Explain the performance you achieved and your justify your decisions.


answer: O desempenho do modelo instruído na classificação de sentimentos foi avaliado com base na precisão e na consistência das previsões. O modelo demonstrou uma forte capacidade de compreender o sentimento contextual, especialmente em cenários de few-shot. No entanto, ocorreram algumas classificações incorretas em casos de sarcasmo ou frases ambíguas.

Para melhorar o desempenho, foi utilizado o few-shot prompting com três exemplos, o que resultou no melhor equilíbrio entre precisão e eficiência. Além disso, a otimização da estrutura dos prompts, como a reformulação das instruções e a adição de detalhes clarificadores, contribuiu para melhores resultados. A decisão de utilizar um modelo instruído foi justificada pela sua maior capacidade de seguir prompts estruturados em comparação com os modelos base. Foram utilizadas métricas como precisão (85%) e F1-score (0,82) para medir o desempenho do modelo.